# 03 - Embeddings Tradicionais e Comparacao

Neste notebook geramos embeddings com `HOG`, visualizamos os resultados em 2D e salvamos a configuracao padrao do extrator considerando **as imagens originais e as imagens sinteticas**.


In [1]:
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from math import ceil
from skimage.color import rgb2gray
from skimage.feature import hog
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    import umap
except ImportError:
    umap = None

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
SYNTHETIC_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'stylegan2_synthetic_256'
HOG_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_hog.parquet'
DEEP_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_deep_resnet50.parquet'
MODEL_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'hog_extractor_config.pkl'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
SAMPLE_FRACTION = 0.15
SAMPLE_RANDOM_SEED = 42
MAX_SAMPLES_PER_SOURCE = None
IMAGE_SIZE = 64
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)

for folder in [HOG_OUTPUT_PATH.parent, MODEL_OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Originais em:', DATASET_ROOT)
print('Sinteticas em:', SYNTHETIC_ROOT)
print('Fracao da amostra:', SAMPLE_FRACTION)
print('Sample random seed:', SAMPLE_RANDOM_SEED)
print('Maximo por fonte:', MAX_SAMPLES_PER_SOURCE)
print('Tamanho da imagem para HOG:', IMAGE_SIZE)


Originais em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba
Sinteticas em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/processed/stylegan2_synthetic_256
Fracao da amostra: 0.15
Sample random seed: 42
Maximo por fonte: None
Tamanho da imagem para HOG: 64


In [3]:
def build_combined_manifest(original_root, synthetic_root=None, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
    sources = [
        ('original', Path(original_root), '*.jpg'),
        ('synthetic', Path(synthetic_root), '*.png'),
    ]

    frames = []
    for source_name, root, pattern in sources:
        if root is None or not root.exists():
            continue
        images = sorted(root.glob(pattern))
        if images:
            sample_size = min(len(images), ceil(len(images) * sample_fraction))
            rng = random.Random(sample_random_seed)
            images = sorted(rng.sample(images, sample_size))
        if max_samples_per_source:
            images = images[:max_samples_per_source]
        if images:
            frames.append(pd.DataFrame({
                'image_name': [path.name for path in images],
                'image_path': [str(path) for path in images],
                'source': source_name,
            }))

    if not frames:
        raise FileNotFoundError('Nenhuma imagem original ou sintetica foi encontrada.')

    return pd.concat(frames, ignore_index=True)

def estimate_hog_feature_count(image_size=64, pixels_per_cell=(8, 8), cells_per_block=(2, 2), orientations=9):
    cells_y = image_size // pixels_per_cell[0]
    cells_x = image_size // pixels_per_cell[1]
    blocks_y = cells_y - cells_per_block[0] + 1
    blocks_x = cells_x - cells_per_block[1] + 1
    if blocks_y <= 0 or blocks_x <= 0:
        raise ValueError('A configuracao do HOG e invalida para o tamanho da imagem informado.')
    return blocks_y * blocks_x * cells_per_block[0] * cells_per_block[1] * orientations

def extract_hog_embeddings(original_root, synthetic_root, output_path, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None, image_size=64, pixels_per_cell=(8, 8), cells_per_block=(2, 2), batch_size=512, max_ram_mb=768):
    manifest = build_combined_manifest(
        original_root=original_root,
        synthetic_root=synthetic_root,
        sample_fraction=sample_fraction,
        sample_random_seed=sample_random_seed,
        max_samples_per_source=max_samples_per_source,
    )
    feature_count = estimate_hog_feature_count(
        image_size=image_size,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
    )
    estimated_ram_mb = (len(manifest) * feature_count * np.dtype(np.float32).itemsize) / (1024 ** 2)
    print(f'Extraindo HOG para {len(manifest)} imagens em {image_size}x{image_size}...')
    print(f'Cada embedding tera {feature_count:,} features.')
    print(f'Estimativa minima para a matriz completa em float32: {estimated_ram_mb:,.1f} MB')
    if estimated_ram_mb > max_ram_mb:
        raise MemoryError(
            'A configuracao atual do HOG deve exceder a memoria do notebook. '
            f'Reduza IMAGE_SIZE ou SAMPLE_FRACTION. Estimativa atual: {estimated_ram_mb:,.1f} MB.'
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    feature_columns = [f'f_{index:04d}' for index in range(feature_count)]
    metadata_rows = []
    feature_rows = []
    batches = []

    def flush_batch(batch_number):
        if not feature_rows:
            return None
        metadata_frame = pd.DataFrame(metadata_rows)
        feature_frame = pd.DataFrame(np.vstack(feature_rows), columns=feature_columns, dtype='float32')
        batch_frame = pd.concat([metadata_frame, feature_frame], axis=1)
        batch_path = output_path.with_name(f'{output_path.stem}_part_{batch_number:03d}{output_path.suffix}')
        batch_frame.to_parquet(batch_path, index=False)
        metadata_rows.clear()
        feature_rows.clear()
        return batch_path

    for index, row in enumerate(manifest.itertuples(index=False), start=1):
        with Image.open(row.image_path) as image:
            image = image.convert('RGB').resize((image_size, image_size))
            gray = rgb2gray(np.asarray(image))
        embedding = hog(gray, pixels_per_cell=pixels_per_cell, cells_per_block=cells_per_block, feature_vector=True).astype(np.float32)
        metadata_rows.append({'image_name': row.image_name, 'image_path': row.image_path, 'source': row.source})
        feature_rows.append(embedding)
        if len(feature_rows) >= batch_size:
            batch_path = flush_batch(len(batches))
            if batch_path is not None:
                batches.append(batch_path)
        if index % 250 == 0 or index == len(manifest):
            print(f'  processadas {index}/{len(manifest)} imagens')
    batch_path = flush_batch(len(batches))
    if batch_path is not None:
        batches.append(batch_path)
    frame = pd.concat([pd.read_parquet(batch_path) for batch_path in batches], ignore_index=True)
    frame.to_parquet(output_path, index=False)
    for batch_path in batches:
        batch_path.unlink(missing_ok=True)
    return frame

def feature_matrix(frame):
    return frame.drop(columns=['image_name', 'image_path', 'source'])

def reduce_embeddings(frame, method='pca', random_state=42):
    features = feature_matrix(frame)
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')
    reduced = reducer.fit_transform(features)
    result = frame[['image_name', 'image_path', 'source']].copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def save_scatter_plot(frame, output_path, title, hue='source'):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=frame, x='x', y='y', hue=hue, alpha=0.75, s=40)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def save_pickle_model(model, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as file:
        pickle.dump(model, file)
    return output_path

def build_hog_extractor_config(image_size=64, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    return {
        'method': 'hog',
        'image_size': image_size,
        'pixels_per_cell': pixels_per_cell,
        'cells_per_block': cells_per_block,
    }


## Embeddings HOG

In [4]:
hog_df = extract_hog_embeddings(
    original_root=DATASET_ROOT,
    synthetic_root=SYNTHETIC_ROOT,
    output_path=HOG_OUTPUT_PATH,
    sample_fraction=SAMPLE_FRACTION,
    sample_random_seed=SAMPLE_RANDOM_SEED,
    max_samples_per_source=MAX_SAMPLES_PER_SOURCE,
    image_size=IMAGE_SIZE,
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
)
hog_df.head()


Extraindo HOG para 34037 imagens em 64x64...
Cada embedding tera 1,764 features.
Estimativa minima para a matriz completa em float32: 229.0 MB
  processadas 250/34037 imagens
  processadas 500/34037 imagens
  processadas 750/34037 imagens
  processadas 1000/34037 imagens
  processadas 1250/34037 imagens
  processadas 1500/34037 imagens
  processadas 1750/34037 imagens
  processadas 2000/34037 imagens
  processadas 2250/34037 imagens
  processadas 2500/34037 imagens
  processadas 2750/34037 imagens
  processadas 3000/34037 imagens
  processadas 3250/34037 imagens
  processadas 3500/34037 imagens
  processadas 3750/34037 imagens
  processadas 4000/34037 imagens
  processadas 4250/34037 imagens
  processadas 4500/34037 imagens
  processadas 4750/34037 imagens
  processadas 5000/34037 imagens
  processadas 5250/34037 imagens
  processadas 5500/34037 imagens
  processadas 5750/34037 imagens
  processadas 6000/34037 imagens
  processadas 6250/34037 imagens
  processadas 6500/34037 imagens
  

,image_name,image_path,source,f_0000,f_0001,f_0002,f_0003,f_0004,f_0005,f_0006,...,f_1754,f_1755,f_1756,f_1757,f_1758,f_1759,f_1760,f_1761,f_1762,f_1763
0,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.005359,0.000000,0.004477,0.444008,0.444008,0.001798,0.000000,...,0.000000,0.088292,0.089092,0.137148,0.185247,0.170362,0.227934,0.200743,0.069498,0.053144
1,000010.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.118572,0.003726,0.002354,0.000000,0.009986,0.011160,0.018830,...,0.013900,0.211435,0.022234,0.120581,0.023699,0.086977,0.190640,0.006868,0.012109,0.028321
2,000016.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.068156,0.104249,0.160879,0.000000,0.303398,0.326847,0.253116,0.216008,0.157118,0.099468
3,000023.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.092914,0.002723,0.004446,0.114142,0.442103,0.050367,0.048261,...,0.079357,0.006914,0.000000,0.010698,0.046744,0.336982,0.336982,0.000000,0.000000,0.000000
4,000025.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.133586,0.015734,0.000000,0.056191,0.311791,0.262462,0.243309,...,0.046329,0.106607,0.292623,0.192189,0.046286,0.089101,0.153109,0.298771,0.298771,0.028891


In [5]:
hog_df = pd.read_parquet(HOG_OUTPUT_PATH)
hog_2d = reduce_embeddings(hog_df, method='pca')
hog_2d.head()


,image_name,image_path,source,x,y,method
0,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.555235,2.112849,pca
1,000010.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.233911,-0.557755,pca
2,000016.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,-0.656634,0.557217,pca
3,000023.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.350784,-0.296675,pca
4,000025.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,-0.925923,0.772837,pca


In [6]:
figure_path = FIGURES_DIR / 'celeba_align_hog_pca.png'
save_scatter_plot(hog_2d, figure_path, 'CelebA Align - HOG (PCA)')
figure_path

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/reports/figures/celeba_align_hog_pca.png')

In [7]:
hog_config = build_hog_extractor_config(
    image_size=IMAGE_SIZE,
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
)
model_path = save_pickle_model(hog_config, MODEL_OUTPUT_PATH)
print('Configuracao padrao do HOG salva em:', model_path)

Configuracao padrao do HOG salva em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/artifacts/models/hog_extractor_config.pkl


## Comparacao entre deep e HOG

In [8]:
hog_features = feature_matrix(hog_df)
hog_labels = (hog_df['source'] == 'synthetic').astype(int)

X_train_hog, X_test_hog, y_train_hog, y_test_hog = train_test_split(
    hog_features,
    hog_labels,
    test_size=0.20,
    random_state=SAMPLE_RANDOM_SEED,
    stratify=hog_labels,
)

hog_tree_classifier = DecisionTreeClassifier(
    random_state=SAMPLE_RANDOM_SEED,
    max_depth=12,
    min_samples_leaf=5,
)
hog_tree_classifier.fit(X_train_hog, y_train_hog)
hog_predictions = hog_tree_classifier.predict(X_test_hog)
hog_accuracy = accuracy_score(y_test_hog, hog_predictions)

hog_metrics = {
    'metric': 'source_classification',
    'classifier': 'decision_tree',
    'accuracy': float(hog_accuracy),
    'error_rate': float(1.0 - hog_accuracy),
    'train_samples': int(len(X_train_hog)),
    'test_samples': int(len(X_test_hog)),
}

print('Classificador de arvore para embeddings HOG: DecisionTreeClassifier')
print('Accuracy HOG (original vs synthetic):', round(hog_metrics['accuracy'], 4))
print('Error rate HOG:', round(hog_metrics['error_rate'], 4))
pd.DataFrame([hog_metrics])


Classificador de arvore para embeddings HOG: DecisionTreeClassifier
Accuracy HOG (original vs synthetic): 0.9697
Error rate HOG: 0.0303


,metric,classifier,accuracy,error_rate,train_samples,test_samples
0,source_classification,decision_tree,0.969741,0.030259,27229,6808


In [9]:
if DEEP_OUTPUT_PATH.exists():
    deep_df = pd.read_parquet(DEEP_OUTPUT_PATH)
    deep_features = feature_matrix(deep_df)
    deep_labels = (deep_df['source'] == 'synthetic').astype(int)

    X_train_deep, X_test_deep, y_train_deep, y_test_deep = train_test_split(
        deep_features,
        deep_labels,
        test_size=0.20,
        random_state=SAMPLE_RANDOM_SEED,
        stratify=deep_labels,
    )

    deep_tree_classifier = DecisionTreeClassifier(
        random_state=SAMPLE_RANDOM_SEED,
        max_depth=12,
        min_samples_leaf=5,
    )
    deep_tree_classifier.fit(X_train_deep, y_train_deep)
    deep_predictions = deep_tree_classifier.predict(X_test_deep)
    deep_accuracy = accuracy_score(y_test_deep, deep_predictions)

    deep_metrics = {
        'metric': 'source_classification',
        'classifier': 'decision_tree',
        'accuracy': float(deep_accuracy),
        'error_rate': float(1.0 - deep_accuracy),
        'train_samples': int(len(X_train_deep)),
        'test_samples': int(len(X_test_deep)),
    }

    comparison_df = pd.DataFrame([
        {'embedding': 'deep_resnet50', **deep_metrics},
        {'embedding': 'hog', **hog_metrics},
    ])
    print('Comparacao de acuracia entre HOG e ResNet50 com DecisionTreeClassifier:')
    comparison_df[['embedding', 'classifier', 'accuracy', 'error_rate', 'train_samples', 'test_samples']]
else:
    print('Dados de comparacao nao disponiveis. Execute o notebook 03 para gerar os embeddings profundos combinando imagens originais e sinteticas.')

Comparacao de acuracia entre HOG e ResNet50 com DecisionTreeClassifier:
